In [7]:
import ollama
import requests
import json
import re
from slot_fillers import fill_appointment_slots

In [8]:
slot_tracker = {    "name": None,
                    "date": None,
                    "time": None,
                    "department": None 
               }
slot_filling_mode = False        

In [9]:
def get_intent_from_ollama(user_input):
    prompt = f"""
You are an intent detection system for a healthcare chatbot. 

Classify the user's intent into one of the following categories:
- greeting
- appointment_booking
- appointment_cancel
- general_inquiry
- wellness_recommendation
- unknown

User input: "{user_input}"

Reply with only the intent name.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3", 
            "prompt": prompt,
            "stream": False
        }
    )

    result = response.json()
    return result['response'].strip().lower()

In [10]:
intent = get_intent_from_ollama("I wanna book an appointment")
print(intent)

appointment_booking


In [11]:
system_prompt = """
You are a helpful healthcare assistant. 
Given a user message, identify the user's intent and reply with a helpful response.
Return the result in JSON format:
{
  "intent": "<intent_name>",
  "response": "<your natural lan
  guage reply>"
}

Intents can include: greeting, book_appointment, reschedule_appointment, health_advice, faq, goodbye.
Do not include diagnoses or prescriptions.
"""


In [13]:
def process_user_message(user_input):
    global slot_filling_mode
     # First check if we are in slot filling mode
    if slot_filling_mode:
        complete, response = fill_appointment_slots(user_input, slot_tracker)
        if complete:
            slot_filling_mode = False  # Done filling
        return response
     # Otherwise, detect intent and proceed accordingly
    intent = get_intent_from_ollama(user_input)

    if intent == 'appointment_booking':
        slot_filling_mode = True
        return fill_appointment_slots(user_input, slot_tracker)[1]
     
    # Regular system response for other intents
    system = system_prompt
    prompt = f"User: {user_input}"

    response = ollama.chat(model='llama3', messages=[
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': prompt}
    ])

    content = response['message']['content']

   
    try:
        match = re.search(r'\{.*\}', content, re.DOTALL)
        if match:
            json_response = json.loads(match.group())
            return json_response['response']
        else:
            raise ValueError("No valid JSON object found")
    except Exception as e:
        print("Error parsing response:", e)
        print("Raw response:", content)
        return "Sorry, I had trouble understanding that"


In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    bot_response = process_user_message(user_input)
    print("Bot:", bot_response)


You:  I wanna book an appointment
